In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
#CREATE MASK

import os
import json
import cv2
import numpy as np
import pandas as pd
from shapely import wkt
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import shutil

DAMAGE_MAPPING = {
    "un-classified": 255,
    "no-damage": 1,
    "minor-damage": 2,
    "major-damage": 3,
    "destroyed": 4
}

def process_polygons(json_path, mask_canvas, is_post=False):
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)
        bldgs = data['features']['xy']
        for bldg in bldgs:
            wkt_polygon = bldg['wkt']
            poly = wkt.loads(wkt_polygon)
            coords = np.array(poly.exterior.coords, dtype=np.int32)
            if is_post:
                subtype = bldg['properties'].get('subtype', 'un-classified')
                pixel_val = DAMAGE_MAPPING.get(subtype, 255)
            else:
                pixel_val = 1
            cv2.fillPoly(mask_canvas, [coords], pixel_val)
    except Exception:
        pass
    return mask_canvas

def process_single_row(idx, row, masks_dir):
    try:
        pre_img_path = row['pre_img_path']
        img = cv2.imread(pre_img_path)
        if img is None:
            return idx, None, None

        h, w = img.shape[:2]
        pre_mask = np.zeros((h, w), dtype=np.uint8)
        post_mask = np.zeros((h, w), dtype=np.uint8)

        pre_mask = process_polygons(row['pre_lbl_path'], pre_mask, is_post=False)
        post_mask = process_polygons(row['post_lbl_path'], post_mask, is_post=True)

        pre_name = os.path.basename(row['pre_img_path']).replace('.png', '_mask.png')
        post_name = os.path.basename(row['post_img_path']).replace('.png', '_mask.png')

        save_pre = os.path.join(masks_dir, "pre", pre_name)
        save_post = os.path.join(masks_dir, "post", post_name)

        cv2.imwrite(save_pre, pre_mask)
        cv2.imwrite(save_post, post_mask)

        return idx, save_pre, save_post
    except Exception as e:
        return idx, None, None

def generate_masks_turbo(base_dir, csv_filename):
    csv_path = os.path.join(base_dir, csv_filename)
    df = pd.read_csv(csv_path)

    df['pre_mask_path'] = None
    df['post_mask_path'] = None

    local_masks_dir = "/content/masks"
    os.makedirs(os.path.join(local_masks_dir, "pre"), exist_ok=True)
    os.makedirs(os.path.join(local_masks_dir, "post"), exist_ok=True)

    valid_indices = df[(df['is_clean'] == True) & (df['is_corrupted'] == False)].index
    print(f"Processing a total of {len(valid_indices)} data points...")

    with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
        futures = {executor.submit(process_single_row, idx, df.loc[idx], local_masks_dir): idx for idx in valid_indices}

        for future in tqdm(as_completed(futures), total=len(valid_indices)):
            idx, pre_path, post_path = future.result()
            if pre_path and post_path:
                df.at[idx, 'pre_mask_path'] = pre_path
                df.at[idx, 'post_mask_path'] = post_path

    output_csv = os.path.join(base_dir, "resqmap_xbd_metadata_with_masks.csv")
    df.to_csv(output_csv, index=False)

    print("\nMask production is complete! It's being copied to Drive—please wait a moment...")

    zip_path = "/content/masks_archive"
    shutil.make_archive(zip_path, 'zip', local_masks_dir)
    shutil.copy(f"{zip_path}.zip", os.path.join(base_dir, "masks.zip"))

    print(f"The masks have been saved as masks.zip to the following location: {base_dir}")

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/"
CSV_FILE = "resqmap_xbd_metadata.csv"

generate_masks_turbo(BASE_DIR, CSV_FILE)

!cp /content/resqmap_xbd_metadata_with_masks.csv /content/drive/MyDrive/RESQMAP/data/xBD/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Processing a total of 11025 data points...


100%|██████████| 11025/11025 [05:15<00:00, 34.99it/s]



Mask production is complete! It's being copied to Drive—please wait a moment...
The masks have been saved as masks.zip to the following location: /content/
cp: invalid option -- 'C'
Try 'cp --help' for more information.


In [ ]:
#CREATE METADATA

import os
import json
import pandas as pd
import numpy as np
import rasterio
from tqdm import tqdm
from typing import Optional, List, Dict, Any

def get_file_path(directory: str, base_name: str, extensions: List[str]) -> Optional[str]:
    for ext in extensions:
        path = os.path.join(directory, f"{base_name}{ext}")
        if os.path.exists(path):
            return path
    return None

def generate_xbd_metadata(base_path: str, output_csv: str) -> pd.DataFrame:
    data_list: List[Dict[str, Any]] = []

    splits = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]

    for split in splits:
        images_dir = os.path.join(base_path, split, 'images')
        labels_dir = os.path.join(base_path, split, 'labels')

        if not os.path.exists(images_dir) and not os.path.exists(labels_dir):
            continue

        all_files = []
        if os.path.exists(images_dir):
            all_files.extend(os.listdir(images_dir))
        if os.path.exists(labels_dir):
            all_files.extend(os.listdir(labels_dir))

        unique_prefixes = set()
        for f in all_files:
            if '_pre_disaster' in f:
                unique_prefixes.add(f.split('_pre_disaster')[0])
            elif '_post_disaster' in f:
                unique_prefixes.add(f.split('_post_disaster')[0])

        for prefix in tqdm(sorted(list(unique_prefixes)), desc=f"Indexing Split: {split}"):

            pre_img_path = get_file_path(images_dir, f"{prefix}_pre_disaster", ['.tif', '.png'])
            post_img_path = get_file_path(images_dir, f"{prefix}_post_disaster", ['.tif', '.png'])
            pre_lbl_path = get_file_path(labels_dir, f"{prefix}_pre_disaster", ['.json'])
            post_lbl_path = get_file_path(labels_dir, f"{prefix}_post_disaster", ['.json'])

            has_pre_img = pre_img_path is not None
            has_post_img = post_img_path is not None
            has_pre_lbl = pre_lbl_path is not None
            has_post_lbl = post_lbl_path is not None

            missing_files = []
            if not has_pre_img: missing_files.append("pre_img")
            if not has_post_img: missing_files.append("post_img")
            if not has_pre_lbl: missing_files.append("pre_lbl")
            if not has_post_lbl: missing_files.append("post_lbl")

            error_log = f"Missing files: {', '.join(missing_files)}" if missing_files else None

            disaster_type = 'unknown'
            sensor = 'unknown'
            gsd = 0.0
            bldg_count = 0
            cloud_rate = -1.0
            is_clean = False
            is_corrupted = False
            dmg_counts = {"no-damage": 0, "minor-damage": 0, "major-damage": 0, "destroyed": 0, "un-classified": 0}

            try:
                lbl_path_to_read = pre_lbl_path if has_pre_lbl else post_lbl_path
                if lbl_path_to_read:
                    with open(lbl_path_to_read, 'r', encoding='utf-8') as f:
                        lbl_data = json.load(f)

                    meta_info = lbl_data.get('metadata', {})
                    disaster_type = meta_info.get('disaster_type', 'unknown')
                    sensor = meta_info.get('sensor', 'unknown')
                    gsd = meta_info.get('gsd', 0.0)
                    bldg_count = len(lbl_data.get('features', {}).get('xy', []))

                if has_post_lbl:
                    with open(post_lbl_path, 'r', encoding='utf-8') as f:
                        post_lbl_data = json.load(f)

                    for feat in post_lbl_data.get('features', {}).get('xy', []):
                        dmg_type = feat.get('properties', {}).get('subtype', 'un-classified')
                        if dmg_type in dmg_counts:
                            dmg_counts[dmg_type] += 1
                        else:
                            dmg_counts["un-classified"] += 1

                if has_pre_img:
                    with rasterio.open(pre_img_path) as src:
                        data = src.read(1)
                        max_val = np.max(data) if np.max(data) > 0 else 1
                        cloud_rate = round(float(np.sum(data > (max_val * 0.8)) / data.size), 4)
                        is_clean = bool(np.std(data) > 5)

            except Exception as e:
                is_corrupted = True
                error_log = f"Corrupted file error: {str(e)}"

            q_score = 0.0
            if not is_corrupted and cloud_rate >= 0:
                q_cloud = (1.0 - cloud_rate) * 0.4
                q_bldg = min(bldg_count / 30.0, 1.0) * 0.6
                q_score = round(q_cloud + q_bldg, 2)

            data_list.append({
                'image_id': prefix,
                'split': split,
                'has_pre_img': has_pre_img,
                'has_post_img': has_post_img,
                'has_pre_lbl': has_pre_lbl,
                'has_post_lbl': has_post_lbl,
                'is_corrupted': is_corrupted,
                'error_log': error_log,
                'disaster_type': disaster_type,
                'sensor': sensor,
                'gsd': gsd,
                'cloud_rate': cloud_rate,
                'bldg_count': bldg_count,
                'is_clean': is_clean,
                'no_damage': dmg_counts['no-damage'],
                'minor_damage': dmg_counts['minor-damage'],
                'major_damage': dmg_counts['major-damage'],
                'destroyed': dmg_counts['destroyed'],
                'un_classified': dmg_counts['un-classified'],
                'quality_score': q_score,
                'pre_img_path': pre_img_path,
                'post_img_path': post_img_path,
                'pre_lbl_path': pre_lbl_path,
                'post_lbl_path': post_lbl_path
            })

    df = pd.DataFrame(data_list)
    df.to_csv(output_csv, index=False)
    print(f"\nMetadata created successfuly: {output_csv} | Total records: {len(df)}")
    return df


if __name__ == "__main__":
    BASE_DIR = "/content/geotiffs"
    OUTPUT_CSV = "/content/resqmap_xbd_metadata.csv"

    if not os.path.exists(BASE_DIR):
        print(f"ERROR: '{BASE_DIR}' folder not found!")
    else:
        df_metadata = generate_xbd_metadata(BASE_DIR, output_csv=OUTPUT_CSV)

In [6]:
!sudo apt-get install pigz

from google.colab import drive
drive.mount('/content/drive')

#UNZIP FROM DRIVE TO COLAB LOCAL
try:
  #xview2_geotiff.tgz - xBD full dataset
  !tar -I pigz -xf /content/drive/MyDrive/RESQMAP/data/xBD/xview2_geotiff.tgz -C /content/
  print("xview2_geotiff.tgz done!")
except Exception as e:
  print(e)


try:
  #masks.zip
  !unzip -q /content/drive/MyDrive/RESQMAP/data/xBD/masks.zip -d /content/
  print("mask.zip done!")
except Exception as e:
  print(e)


#COPY METADATA FROM DRIVE TO COLAB LOCAL
!cp /content/drive/MyDrive/RESQMAP/data/xBD/resqmap_xbd_metadata.csv /content/
!cp /content/drive/MyDrive/RESQMAP/data/xBD/resqmap_xbd_metadata_with_masks.csv /content/

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pigz is already the newest version (2.6-1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
